# providers.azure

OpenAI chat completions served from an Azure deployment.

In [ ]:
#| default_exp providers.azure

In [ ]:
#| hide
from nbdev.showdoc import *

## The provider

Construction is cheap and side-effect-free — only `_get_client` reaches for `AZURE_API_KEY`. That keeps `from nbdialog.providers.azure import AzureProvider` safe in environments without credentials (offline doc builds, CI without secrets, tests that mock `complete`).

`complete` is a request/response loop, not a single call. When `tools` are passed in, the model can answer directly *or* emit a `tool_calls` array — in which case we dispatch each call against the registered Python function, append the result as a `role: "tool"` message, and re-call. `max_tool_steps` bounds the loop in case the model gets stuck. When no tools are passed, the loop exits on the first response — identical behavior to the original single-shot version.

Defaults match the constants that used to live in `core` so existing notebooks need exactly one line of setup to keep working:

```python
from nbdialog.providers.azure import AzureProvider
set_provider(AzureProvider())
```

In [ ]:
#| export
import json, os
from openai import AzureOpenAI
from nbdialog.core import Tool

class AzureProvider:
    "OpenAI chat completions via an Azure deployment, with a tool-call loop."
    def __init__(self,
                 deployment: str = "gpt-5.4",
                 endpoint: str = "https://pablo-ml1b1csr-eastus2.cognitiveservices.azure.com",
                 api_version: str = "2024-12-01-preview",
                 api_key_env: str = "AZURE_API_KEY",
                 max_completion_tokens: int = 16384,
                 max_tool_steps: int = 8):
        self.deployment, self.endpoint = deployment, endpoint
        self.api_version, self.api_key_env = api_version, api_key_env
        self.max_completion_tokens = max_completion_tokens
        self.max_tool_steps = max_tool_steps
        self._client = None

    def _get_client(self):
        if self._client is None:
            self._client = AzureOpenAI(api_key=os.environ[self.api_key_env],
                                       azure_endpoint=self.endpoint,
                                       api_version=self.api_version)
        return self._client

    def complete(self, messages: list[dict], tools: list[Tool] = None) -> str:
        tools = tools or []
        schemas = [t.schema for t in tools]
        dispatch = {t.schema["function"]["name"]: t.fn for t in tools}
        msgs = list(messages)
        for _ in range(self.max_tool_steps):
            kw = {"tools": schemas} if schemas else {}
            resp = self._get_client().chat.completions.create(
                model=self.deployment, messages=msgs,
                max_completion_tokens=self.max_completion_tokens, **kw)
            m = resp.choices[0].message
            if not m.tool_calls: return m.content
            msgs.append(m.model_dump(exclude_none=True))
            for tc in m.tool_calls:
                out = dispatch[tc.function.name](**json.loads(tc.function.arguments))
                msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(out)})
        raise RuntimeError(f"Tool loop did not converge in {self.max_tool_steps} steps")

Smoke test — only runs when credentials are present, so the doc build stays green without them.

In [ ]:
if os.environ.get("AZURE_API_KEY"):
    print(AzureProvider().complete([{"role":"user","content":"ping"}]))

pong


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()